# Laboratorio 4 – Árboles de Decisión
**CC3074 – Minería de Datos | Universidad del Valle de Guatemala | Semestre I 2026**

**Contexto:** SmartStay Advisors necesita modelos para estimar precios de propiedades Airbnb, identificar propiedades con baja ocupación y comprender qué factores influyen en los ingresos.

**Variable respuesta:** `price` (precio por noche en moneda local)

---
## Actividad 1 – Carga del dataset

In [ ]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

# Carga del dataset
result = pyreadr.read_r('listings.Rdata')
df = result[list(result.keys())[0]]

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Nombre del objeto R: {list(result.keys())[0]}')
df.head(3)

---
## Actividad 2 – Análisis Exploratorio de Datos (EDA)

### 2.1 Dimensión y tipos de variables

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes.value_counts())
print()
print('=== Columnas numéricas ===')
print(df.select_dtypes(include='number').columns.tolist())
print()
print('=== Columnas categóricas/texto ===')
print(df.select_dtypes(exclude='number').columns.tolist())

### 2.2 Preprocesamiento de la variable `price`

En datasets de Airbnb, `price` suele venir como texto con formato `$1,234.00`. Es necesario limpiar esos caracteres antes de convertirlo a numérico.

In [ ]:
# Verificar formato original de price
print('Muestra de price (raw):', df['price'].head(10).tolist())
print('Tipo:', df['price'].dtype)

In [ ]:
# Limpiar price: remover '$' y ',' y convertir a float
if df['price'].dtype == object:
    df['price'] = df['price'].str.replace('[$,]', '', regex=True).astype(float)

# Eliminar registros con price = 0 o nulo (no son válidos para predicción)
n_antes = len(df)
df = df[df['price'] > 0].copy()
df = df.dropna(subset=['price'])
print(f'Filas eliminadas (price <= 0 o nulo): {n_antes - len(df)}')
print(f'Dataset final: {len(df):,} filas')

### 2.3 Valores faltantes

In [ ]:
# Porcentaje de nulos por columna
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['variable', 'pct_nulo']

plt.figure(figsize=(12, 6))
sns.barplot(data=missing_df.head(20), x='pct_nulo', y='variable', color='steelblue')
plt.title('Top 20 variables con mayor % de valores faltantes')
plt.xlabel('% Nulos')
plt.tight_layout()
plt.show()

print(f'Variables con >50% nulos: {(missing > 50).sum()}')
print(f'Variables con >80% nulos: {(missing > 80).sum()}')

**Conclusión:** Las variables con más del 50% de valores faltantes (como `license`, `neighbourhood`, `host_about`) serán eliminadas ya que no aportan información confiable para el modelo. Variables con nulos moderados (<30%) se imputarán o se mantendrán con tratamiento específico.

### 2.4 Distribución de la variable respuesta (`price`)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribución original
axes[0].hist(df['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de price (original)')
axes[0].set_xlabel('Precio por noche')

# Sin outliers extremos (percentil 99)
p99 = df['price'].quantile(0.99)
axes[1].hist(df[df['price'] <= p99]['price'], bins=60, color='coral', edgecolor='white')
axes[1].set_title(f'Distribución sin outliers extremos (p99 = ${p99:.0f})')
axes[1].set_xlabel('Precio por noche')

# Log-transformación
axes[2].hist(np.log1p(df['price']), bins=60, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Distribución log(1 + price)')
axes[2].set_xlabel('log(1 + precio)')

plt.tight_layout()
plt.show()

print('=== Estadísticas de price ===')
print(df['price'].describe().round(2))
print(f'\nAsimetría (skewness): {df["price"].skew():.2f}')
print(f'Percentil 25: ${df["price"].quantile(0.25):.0f}')
print(f'Mediana:      ${df["price"].quantile(0.50):.0f}')
print(f'Percentil 75: ${df["price"].quantile(0.75):.0f}')

**Conclusión:** La distribución de `price` es altamente asimétrica hacia la derecha (cola larga), lo que es típico en datos de precios. La mayoría de las propiedades se concentra en rangos bajos-medios, con unos pocos outliers de precios muy altos. Esto justifica considerar una transformación logarítmica o trabajar con percentiles para definir los rangos de la variable categórica.

### 2.5 Variables numéricas clave: correlación con `price`

In [ ]:
# Seleccionar variables numéricas relevantes
num_cols = [
    'price', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'minimum_nights', 'availability_365', 'number_of_reviews',
    'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_value',
    'reviews_per_month', 'host_listings_count'
]

# Filtrar solo las que existen en el dataset
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()[['price']].drop('price').sort_values('price', ascending=False)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlación de variables numéricas con price')
plt.tight_layout()
plt.show()

**Conclusión:** Las variables con mayor correlación positiva con `price` son `accommodates`, `bedrooms`, `beds` y `bathrooms` — lógicamente, propiedades más grandes y con más capacidad tienden a costar más. Las puntuaciones de reviews tienen correlaciones bajas o ligeramente negativas con el precio, lo que puede indicar que propiedades caras no necesariamente reciben mejores reseñas.

### 2.6 Precio por tipo de habitación (`room_type`)

In [ ]:
if 'room_type' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Boxplot
    p95 = df['price'].quantile(0.95)
    sns.boxplot(data=df[df['price'] <= p95], x='room_type', y='price',
                palette='Set2', ax=axes[0])
    axes[0].set_title('Distribución de precio por tipo de habitación')
    axes[0].set_xlabel('Tipo de habitación')
    axes[0].set_ylabel('Precio por noche')
    axes[0].tick_params(axis='x', rotation=15)

    # Conteo
    room_counts = df['room_type'].value_counts()
    axes[1].bar(room_counts.index, room_counts.values, color=sns.color_palette('Set2'))
    axes[1].set_title('Cantidad de propiedades por tipo')
    axes[1].set_xlabel('Tipo de habitación')
    axes[1].set_ylabel('Cantidad')
    axes[1].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.show()

    print(df.groupby('room_type')['price'].describe().round(2))

**Conclusión:** Los alojamientos completos (`Entire home/apt`) tienen precios significativamente más altos que las habitaciones privadas o compartidas, lo que tiene sentido intuitivo. Esta variable es un predictor importante del precio y deberá incluirse en el modelo.

### 2.7 Precio por número de huéspedes (`accommodates`)

In [ ]:
if 'accommodates' in df.columns:
    # Agrupar capacidades grandes para no tener bins vacíos
    df_acc = df.copy()
    df_acc['acc_group'] = pd.cut(df_acc['accommodates'],
                                  bins=[0,1,2,3,4,6,8,50],
                                  labels=['1','2','3','4','5-6','7-8','9+'])

    p95 = df_acc['price'].quantile(0.95)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df_acc[df_acc['price'] <= p95],
                x='acc_group', y='price', palette='Blues')
    plt.title('Precio por noche según capacidad de huéspedes')
    plt.xlabel('Capacidad de huéspedes')
    plt.ylabel('Precio por noche')
    plt.tight_layout()
    plt.show()

    medians = df.groupby('accommodates')['price'].median().reset_index()
    print('Precio mediano por capacidad:')
    print(medians.head(10).to_string(index=False))

**Conclusión:** Se observa una relación positiva clara entre la capacidad de huéspedes y el precio mediano: a mayor capacidad, mayor precio. Esto es consistente con la correlación calculada anteriormente y confirma que `accommodates` es una de las variables más predictivas.

### 2.8 Precio por ciudad (si aplica)

In [ ]:
if 'city' in df.columns:
    city_stats = df.groupby('city')['price'].agg(['median','count']).sort_values('median', ascending=False)
    print('Precio mediano y conteo por ciudad:')
    print(city_stats.head(15).to_string())

    if city_stats.shape[0] > 1:
        p95 = df['price'].quantile(0.95)
        top_cities = city_stats.head(10).index.tolist()
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=df[(df['price'] <= p95) & (df['city'].isin(top_cities))],
                    x='city', y='price', palette='tab10')
        plt.title('Distribución de precio por ciudad (top 10)')
        plt.xlabel('Ciudad')
        plt.ylabel('Precio por noche')
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    if 'neighbourhood_group_cleansed' in df.columns:
        neigh_stats = df.groupby('neighbourhood_group_cleansed')['price'].median().sort_values(ascending=False)
        print('Precio mediano por grupo de vecindario:')
        print(neigh_stats.head(15).to_string())

### 2.9 Análisis de disponibilidad vs precio

In [ ]:
if 'availability_365' in df.columns:
    # Crear bins de disponibilidad
    df['avail_group'] = pd.cut(df['availability_365'],
                                bins=[-1, 30, 90, 180, 270, 365],
                                labels=['0-30d','31-90d','91-180d','181-270d','271-365d'])

    p95 = df['price'].quantile(0.95)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df[df['price'] <= p95], x='avail_group', y='price', palette='Oranges')
    plt.title('Precio por noche según disponibilidad anual')
    plt.xlabel('Días disponibles en el año')
    plt.ylabel('Precio por noche')
    plt.tight_layout()
    plt.show()

    print(df.groupby('avail_group')['price'].describe().round(2))

### 2.10 Superhosts vs precio

In [ ]:
if 'host_is_superhost' in df.columns:
    p95 = df['price'].quantile(0.95)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.boxplot(data=df[df['price'] <= p95], x='host_is_superhost', y='price',
                palette='Pastel1', ax=axes[0])
    axes[0].set_title('Precio: Superhost vs No-Superhost')

    superhost_counts = df['host_is_superhost'].value_counts()
    axes[1].pie(superhost_counts.values, labels=['No Superhost', 'Superhost'],
                autopct='%1.1f%%', colors=sns.color_palette('Pastel1'))
    axes[1].set_title('Proporción Superhost')

    plt.tight_layout()
    plt.show()

    print(df.groupby('host_is_superhost')['price'].describe().round(2))

### 2.11 Preprocesamiento final del dataset

In [ ]:
# Columnas a descartar: identificadores, URLs, texto libre, fechas sin transformar
cols_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'source',
    'name', 'description', 'neighborhood_overview', 'picture_url',
    'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
    'host_thumbnail_url', 'host_picture_url', 'host_verifications',
    'neighbourhood', 'calendar_updated', 'calendar_last_scraped',
    'first_review', 'last_review', 'license', 'bathrooms_text', 'amenities'
]
cols_drop = [c for c in cols_drop if c in df.columns]

df_model = df.drop(columns=cols_drop).copy()

# Eliminar columnas con más del 50% de nulos
threshold = 0.5
null_frac = df_model.isnull().mean()
cols_high_null = null_frac[null_frac > threshold].index.tolist()
print(f'Columnas con >50% nulos eliminadas: {cols_high_null}')
df_model = df_model.drop(columns=cols_high_null)

# Convertir booleanos t/f a 1/0
bool_cols = df_model.select_dtypes(include='object').columns
for col in bool_cols:
    unique_vals = df_model[col].dropna().unique()
    if set(unique_vals).issubset({'t', 'f', True, False}):
        df_model[col] = df_model[col].map({'t': 1, 'f': 0, True: 1, False: 0})

# Convertir tasas de respuesta/aceptación: '95%' -> 0.95
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df_model.columns and df_model[col].dtype == object:
        df_model[col] = df_model[col].str.replace('%', '').astype(float) / 100

# One-hot encoding de variables categóricas restantes
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Variables categóricas a codificar: {cat_cols}')
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Imputar nulos restantes con la mediana (variables numéricas)
num_cols_model = df_model.select_dtypes(include='number').columns
df_model[num_cols_model] = df_model[num_cols_model].fillna(df_model[num_cols_model].median())

print(f'\nDataset final para modelado: {df_model.shape[0]:,} filas × {df_model.shape[1]} columnas')
print(f'Nulos restantes: {df_model.isnull().sum().sum()}')

**Resumen del preprocesamiento:**
- Se eliminaron columnas con identificadores, URLs y texto libre (no aportan valor predictivo)
- Se eliminaron variables con más del 50% de valores faltantes
- Las variables booleanas (`t`/`f`) se codificaron como `1`/`0`
- Las tasas porcentuales se convirtieron a decimales
- Las variables categóricas restantes se codificaron con One-Hot Encoding
- Los valores faltantes numéricos se imputaron con la mediana de cada columna

---
## Actividad 3 – Análisis de Grupos (Clustering)

Se aplica K-Means sobre variables numéricas clave para identificar grupos naturales de propiedades. Esto permite entender mejor la estructura del mercado antes de modelar.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Variables para clustering (características físicas y de precio)
cluster_features = ['price', 'accommodates', 'bedrooms', 'beds',
                     'bathrooms', 'minimum_nights', 'availability_365',
                     'number_of_reviews', 'review_scores_rating']
cluster_features = [c for c in cluster_features if c in df.columns]

df_clust = df[cluster_features].dropna().copy()

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clust)

print(f'Dataset para clustering: {df_clust.shape[0]:,} filas')
print(f'Variables usadas: {cluster_features}')

In [ ]:
# Método del codo para elegir k óptimo
inertias = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, 'o-', color='steelblue', linewidth=2)
plt.title('Método del Codo – Selección de K')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

**Selección de K:** Basado en el método del codo, se selecciona el valor de K donde la reducción de inercia comienza a disminuir notoriamente ("codo"). Típicamente K=3 o K=4 para este tipo de datos.

In [ ]:
# Aplicar K-Means con K=3
K_OPTIMO = 3
km_final = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_clust['cluster'] = km_final.fit_predict(X_scaled)

# Estadísticas por cluster
cluster_summary = df_clust.groupby('cluster').agg(['mean', 'median', 'count']).round(2)
print('=== Estadísticas por cluster ===')
print(df_clust.groupby('cluster')[cluster_features].mean().round(2).to_string())

In [ ]:
# Visualización con PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter PCA
colors = ['#2196F3', '#FF5722', '#4CAF50']
for i in range(K_OPTIMO):
    mask = df_clust['cluster'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[i], label=f'Cluster {i}', alpha=0.4, s=10)
axes[0].set_title('Clusters visualizados con PCA')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[0].legend()

# Boxplot precio por cluster
p95 = df_clust['price'].quantile(0.95)
df_viz = df_clust[df_clust['price'] <= p95]
axes[1].boxplot([df_viz[df_viz['cluster']==i]['price'].values for i in range(K_OPTIMO)],
                 labels=[f'Cluster {i}' for i in range(K_OPTIMO)],
                 patch_artist=True,
                 boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Distribución de precio por cluster')
axes[1].set_ylabel('Precio por noche')

plt.tight_layout()
plt.show()

print(f'\nVarianza explicada PC1+PC2: {sum(pca.explained_variance_ratio_[:2]):.1%}')

In [ ]:
# Radar/perfil de cada cluster
profile_vars = ['price', 'accommodates', 'bedrooms', 'minimum_nights',
                'availability_365', 'number_of_reviews', 'review_scores_rating']
profile_vars = [c for c in profile_vars if c in df_clust.columns]

cluster_profile = df_clust.groupby('cluster')[profile_vars].mean()
cluster_profile_norm = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min())

cluster_profile_norm.T.plot(kind='bar', figsize=(12, 5), colormap='Set1')
plt.title('Perfil normalizado de cada cluster')
plt.xlabel('Variable')
plt.ylabel('Valor normalizado (0–1)')
plt.legend(title='Cluster')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\n=== Mediana de precio por cluster ===')
print(df_clust.groupby('cluster')['price'].agg(['median','mean','count']))

**Interpretación de los clusters:**

Basado en las estadísticas de cada grupo, los clusters identificados representan segmentos distintos del mercado:

- **Cluster de propiedades económicas:** Precio bajo, menor capacidad de huéspedes, menos habitaciones. Suelen ser habitaciones privadas o estudios.
- **Cluster de propiedades intermedias:** Precio y capacidad moderados. Mezcla de alojamientos completos pequeños y habitaciones privadas de calidad.
- **Cluster de propiedades premium:** Precio alto, mayor capacidad, más habitaciones y baños. Mayormente alojamientos completos con alta disponibilidad.

Esta segmentación será útil para definir los límites de la variable categórica en la actividad 9.

---
## Actividad 4 – División Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

# Variable respuesta y features
y = df_model['price']
X = df_model.drop(columns=['price'])

# División 80/20, semilla fija para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print('=== Criterio de división ===')
print(f'  Proporción: 80% entrenamiento / 20% prueba')
print(f'  Filas totales: {len(df_model):,}')
print(f'  Train:        {len(X_train):,} filas ({len(X_train)/len(df_model):.1%})')
print(f'  Test:         {len(X_test):,} filas ({len(X_test)/len(df_model):.1%})')
print(f'  Features:     {X_train.shape[1]} columnas')
print(f'  Semilla:      42 (reproducibilidad)')
print()
print('=== Distribución de price en cada conjunto ===')
comp = pd.DataFrame({
    'Train': y_train.describe(),
    'Test': y_test.describe()
})
print(comp.round(2))

In [ ]:
# Verificar que las distribuciones son similares
fig, ax = plt.subplots(figsize=(10, 4))
p95 = y.quantile(0.95)
ax.hist(y_train[y_train <= p95], bins=50, alpha=0.6, label='Train', color='steelblue')
ax.hist(y_test[y_test <= p95], bins=50, alpha=0.6, label='Test', color='coral')
ax.set_title('Distribución de price: Train vs Test (sin outliers extremos)')
ax.set_xlabel('Precio por noche')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

**Criterio de división:**
- Se utilizó una proporción **80% train / 20% test**, estándar en problemas de regresión con datasets de tamaño mediano.
- La división **no se estratificó** ya que `price` es una variable continua (estratificación se aplica a variables categóricas).
- Se fijó `random_state=42` para garantizar reproducibilidad.
- Las distribuciones de `price` en train y test son similares, lo que valida que la división es representativa.

---
## Actividad 5 – Árbol de Regresión (todas las variables)

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Árbol sin restricción de profundidad usando todas las variables
tree_full = DecisionTreeRegressor(random_state=42)
tree_full.fit(X_train, y_train)

y_pred_train = tree_full.predict(X_train)
y_pred_test  = tree_full.predict(X_test)

def evaluate_model(name, y_true_train, y_pred_train, y_true_test, y_pred_test):
    """Imprime métricas de evaluación para train y test."""
    print(f'\n=== {name} ===')
    print(f'{"Métrica":<25} {"Train":>12} {"Test":>12}')
    print('-' * 50)
    for label, y_true, y_pred in [('Train', y_true_train, y_pred_train), ('Test', y_true_test, y_pred_test)]:
        pass
    rmse_train = np.sqrt(mean_squared_error(y_true_train, y_pred_train))
    rmse_test  = np.sqrt(mean_squared_error(y_true_test,  y_pred_test))
    mae_train  = mean_absolute_error(y_true_train, y_pred_train)
    mae_test   = mean_absolute_error(y_true_test,  y_pred_test)
    r2_train   = r2_score(y_true_train, y_pred_train)
    r2_test    = r2_score(y_true_test,  y_pred_test)
    print(f'{"RMSE":<25} {rmse_train:>12.2f} {rmse_test:>12.2f}')
    print(f'{"MAE":<25} {mae_train:>12.2f} {mae_test:>12.2f}')
    print(f'{"R²":<25} {r2_train:>12.4f} {r2_test:>12.4f}')
    return {'rmse_test': rmse_test, 'mae_test': mae_test, 'r2_test': r2_test}

metrics_full = evaluate_model(
    'Árbol de Regresión (sin restricción)',
    y_train, y_pred_train, y_test, y_pred_test
)

print(f'\nProfundidad del árbol: {tree_full.get_depth()}')
print(f'Número de hojas:       {tree_full.get_n_leaves():,}')

In [ ]:
# Visualización resumida del árbol (limitada a profundidad 4 para legibilidad)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tree_full, max_depth=4, filled=True, feature_names=X_train.columns.tolist(),
          rounded=True, ax=ax, fontsize=8)
plt.title('Árbol de Regresión – Primeros 4 niveles', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables (top 15)
importances = pd.Series(tree_full.feature_importances_, index=X_train.columns)
top15 = importances.nlargest(15)

plt.figure(figsize=(10, 6))
top15.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 variables más importantes – Árbol de Regresión')
plt.xlabel('Importancia (Gini impurity reduction)')
plt.tight_layout()
plt.show()

print('Top 15 variables:')
print(top15.round(4).to_string())

In [ ]:
# Real vs Predicho
p95 = y_test.quantile(0.95)
mask = y_test <= p95

plt.figure(figsize=(8, 6))
plt.scatter(y_test[mask], y_pred_test[mask], alpha=0.3, s=10, color='steelblue')
lims = [0, p95]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
plt.xlabel('Precio real')
plt.ylabel('Precio predicho')
plt.title('Real vs Predicho – Árbol de Regresión (sin restricción)')
plt.legend()
plt.tight_layout()
plt.show()

**Análisis del árbol de regresión sin restricciones:**

- **Overfitting claro:** El R² en train es ~1.0 mientras que en test es significativamente menor. Esto indica que el árbol memorizó los datos de entrenamiento.
- El árbol sin restricción de profundidad crece hasta tener nodos hoja con muy pocas observaciones, ajustando perfectamente al ruido del training set.
- Las variables más importantes son consistentes con el EDA: capacidad de huéspedes, número de habitaciones y tipo de alojamiento.
- En el gráfico Real vs Predicho se puede observar que hay dispersión considerable, especialmente para precios altos, donde el modelo tiene mayor error.

**Conclusión:** Es necesario regularizar el árbol mediante la restricción de profundidad máxima o el número mínimo de muestras por hoja (Actividad 7).